# 08 — Earnings-Event-Conditional VaR (Fusion of Both Dissertations)

**This notebook is the synthesis point of this whole project.**

It combines:
- The **regime-switching mechanic** from my BSc dissertation's Hybrid Market-and-Climate VaR (`src/hybrid_var.py`) — a scheduled event shifts the simulated return distribution, rather than treating every day identically.
- The **evaluation discipline** from my MSc/Citibank dissertation, *"Reading Between the Lines: Can an LLM Predict How Markets React to Earnings?"* — don't just build an overlay, rigorously test whether it's actually justified. Selectivity vs coverage. Threshold-free signal correlation. Honest reporting of null results.

**What this notebook does:**
1. Scores a small set of sample earnings disclosures for sentiment (lexicon-based proxy — see `src/earnings_signal.py` for the upgrade path to a real LLM call)
2. Runs event-conditional Monte Carlo VaR for each disclosure's scenario
3. Tests whether the score's *direction and magnitude* actually correlates with realised returns (Spearman rank correlation — the strongest evidence type in the Citibank dissertation)
4. Backtests whether the sentiment-adjusted VaR is genuinely better-calibrated than plain VaR on event days — Kupiec and Christoffersen tests, exactly as in `04_backtesting.ipynb`

**Framing, consistent with both dissertations' own conclusions:** this is a risk *model validation* exercise, not an autonomous signal-generation tool. If the overlay doesn't pass backtesting, that is the correct and useful conclusion — not a failure.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.earnings_signal import EarningsSignalScorer
from src.earnings_var import event_conditional_var, signal_return_correlation, backtest_event_overlay

scorer = EarningsSignalScorer()


## 1. Sample disclosures

**Important:** these are short, generic, illustrative sample snippets written for this demo — not real corporate filings or real quotes from any actual company. Replace with real earnings-release text (from investor-relations pages, 8-Ks, or a transcript provider) for a genuine analysis. Structure preserved so real text can be dropped straight in.

In [ ]:
sample_disclosures = {
    'event_1_bullish': (
        'We delivered record revenue growth this quarter, with strong margin '
        'expansion and robust momentum across all segments. Guidance is being '
        'raised given our confidence in continued growth and resilience.'
    ),
    'event_2_bearish': (
        'Revenue declined this quarter, missing expectations amid significant '
        'headwinds and pressure on margins. We are reducing guidance given the '
        'challenging environment and remain cautious about near-term uncertainty.'
    ),
    'event_3_neutral': (
        'The company reported quarterly results in line with prior expectations. '
        'Management discussed ongoing operations and provided a strategic update.'
    ),
    'event_4_mixed': (
        'Revenue growth remained solid, though margins faced pressure from '
        'ongoing headwinds. Management expressed confidence in the long-term '
        'opportunity despite near-term uncertainty.'
    ),
}

scores = {name: scorer.score(text) for name, text in sample_disclosures.items()}
pd.Series(scores, name='sentiment_score')


## 2. Event-conditional VaR per disclosure

Use each disclosure's score to shift the simulated return distribution for that event day, on top of the asset's ordinary (baseline) return distribution.

In [ ]:
baseline_mean, baseline_std = 0.0005, 0.018  # replace with a real asset's historical stats

event_var_results = {
    name: event_conditional_var(baseline_mean, baseline_std, llm_score=score)
    for name, score in scores.items()
}
pd.DataFrame(event_var_results).T.round(4)


## 3. Does the score actually track realised returns?

**This is the single most important test in this notebook** — it directly reproduces the Citibank dissertation's strongest evidence (Spearman rho = 0.2565, p = 0.0001 on 109 real events). Requires real historical (disclosure, realised return) pairs — the cell below shows the call shape on synthetic data; replace with your own collected event history.

In [ ]:
# TODO: replace with real (score, realised_return) pairs collected over time
np.random.seed(1)
n_demo_events = 30
demo_scores = pd.Series(np.random.uniform(-1, 1, n_demo_events))
demo_returns = pd.Series(np.random.normal(0, 0.02, n_demo_events))  # pure noise placeholder

signal_return_correlation(demo_scores, demo_returns)


## 4. Backtest: is the overlay actually better-calibrated than plain VaR?

The key validation question — mirrors the Citibank dissertation's insistence on testing whether added complexity is actually earned by the data, not just assumed. Requires a real history of event-day returns plus both VaR series; shown here with placeholder structure.

In [ ]:
# TODO: replace with real event-day returns and matched plain/overlay VaR series
np.random.seed(2)
n = 30
demo_event_returns = pd.Series(np.random.normal(-0.001, 0.025, n))
demo_plain_var = pd.Series(np.full(n, 0.030))
demo_overlay_var = pd.Series(np.random.uniform(0.025, 0.045, n))

backtest_event_overlay(demo_event_returns, demo_plain_var, demo_overlay_var, confidence=0.95)


## 5. Honest interpretation

*(Once run on real data: does the overlay pass Kupiec/Christoffersen where plain VaR doesn't, or vice versa? Does the Spearman correlation clear statistical significance? Report both outcomes plainly — a null result here is a legitimate, useful finding about where an earnings-aware overlay is and isn't justified, exactly the intellectually honest standard both source dissertations were built on.)*

---

**This completes the fusion of both dissertations into one framework:** BSc regime-switching Hybrid VaR mechanics + MSc LLM-signal validation discipline, applied to a genuinely useful risk-model-validation question. Combined with notebooks 01–07, this project now spans volatility forecasting, Market VaR, backtesting, stress testing, Climate VaR, Hybrid VaR, and this earnings-event overlay — all runnable, testable, and honestly evaluated rather than asserted.